# Vincent-Clean ABC Simulation

This notebook does only one thing: run ABC on 9 small simulated scenarios while respecting Vincent's condition.

For every scenario we check:

1. $\pi_1 > \pi_2 > \cdots > \pi_N$,
2. $z_1/\pi_1 < z_2/\pi_2 < \cdots < z_N/\pi_N$,
3. the CaDSD start `omega=0, rho=0.5` matches the Vincent/P$\pi$ variance.

Then ABC changes `omega` and `rho` to search for a better complex CaDSD kernel. Random search uses the same number of evaluations.

## 1. Settings

Change this cell only.

In [196]:
N = 20
n = 4
BASE_SEED = 4376821 #22363212 #20260023

# Start switch: "vincent" or "random_best"
START_MODE = "vincent"
RANDOM_START_CANDIDATES = 200

# Run settings
MAX_ITERATIONS = 1000
NEEDED_THRESHOLD = 3.5
CHECKPOINT_INTERVAL = 50   
COLONY_SIZE = 10
LIMIT = 20
ONLOOKER_FACTOR = 0.5
LOCAL_SEARCH_INTERVAL = 20
LOCAL_SEARCH_ATTEMPTS = 1

# Validation and mutation settings
STRICT_VALIDATION = True
INITIAL_NEAR_START_SCALE = 0.05
RANDOM_GLOBAL_PROBABILITY = 0.25

## 2. Imports and Core Functions

The functions are included directly here, so the notebook does not need to import the terminal runner.

In [197]:
import time
from dataclasses import dataclass
from typing import Optional, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [198]:
def Ppi(Pi: Union[np.ndarray, list]) -> np.ndarray:
    """
    Python/Numpy port of the R function Ppi(Pi).

    Parameters
    ----------
    Pi : array-like of shape (N,)
        First-order inclusion probabilities, 0 < Pi_i < 1,
        sum(Pi) must be (approximately) an integer.

    Returns
    -------
    V : ndarray of shape (N, n)
        Orthogonal matrix associated to the DSD construction.
    """
    Pi = np.asarray(Pi, dtype=float).ravel()
    N = Pi.size

    # --- Error checks ---
    if N < 2:
        raise ValueError(
            "The sampling designs should be defined on a set of more than "
            "one element. (length(Pi) > 1)"
        )

    if np.any(Pi <= 0) or np.any(Pi >= 1):
        raise ValueError("Pi is not a vector of probabilities (0 < p < 1).")

    sum_pi_rounded = round(Pi.sum(), 9)
    n_int = int(sum_pi_rounded)
    if int(round(sum_pi_rounded, 9)) - sum_pi_rounded != 0:
        raise ValueError(
            "The sum of the first order inclusion probabilities "
            "should be an integer (up to rounding)."
        )

    # --- Main algorithm ---
    s_vals = np.zeros(N, dtype=float)
    c_vals = np.zeros(N, dtype=float)
    alpha = np.zeros(N, dtype=float)

    # kr will store the indices (0-based) where cumulative sum crosses integers
    if n_int <= 0:
        raise ValueError("Sum of Pi must be at least 1.")
    kr = [None] * n_int

    cum_sum = 0.0
    r = 1          # current integer threshold
    r_prev = 0     # last integer that was crossed

    for k in range(N):
        prev_sum = cum_sum
        cum_sum += Pi[k]

        if cum_sum >= r:  # crossed integer r
            if r <= n_int:
                alpha[k] = r - prev_sum
                kr[r - 1] = k   # store 0-based index
                val = np.sqrt((1.0 - Pi[k]) / (1.0 - alpha[k]))
                s_vals[k] = np.round(val, 8)
                r_prev = r
                r += 1
        else:
            denom = (r_prev + 1 - prev_sum)
            val = np.sqrt(Pi[k] / denom)
            s_vals[k] = np.round(val, 15)

        c_vals[k] = np.sqrt(1.0 - s_vals[k] ** 2)

    # Patch: ensure last crossing index corresponds to last unit (like R hack)
    # If some entries of kr are still None, set the last one to N-1.
    if any(x is None for x in kr):
        kr[-1] = N - 1
    # For safety, also replace any remaining None by the last index
    last_index = kr[-1]
    kr = [last_index if x is None else x for x in kr]

    # Use sample size n_int as number of columns (simplified vs. R hack)
    r_prev = n_int

    # --- Build V ---
    V = np.zeros((N, r_prev), dtype=float)
    V[0, 0] = 1.0

    # In R: V[kr[r] + 1, r + 1] = 1 for r in 1:(r_prev-1)
    # Here r_idx corresponds to r-1 in R, and we use 0-based indices.
    if r_prev - 1 != 0:
        for r_idx in range(1, r_prev):
            kpos = kr[r_idx - 1]  # 0-based index for row
            V[kpos + 1, r_idx] = 1.0

    # Apply Givens-like rotations
    for k in range(N - 1):
        L = V[k, :].copy()
        M = V[k + 1, :].copy()
        V[k, :] = s_vals[k] * L - c_vals[k] * M
        V[k + 1, :] = c_vals[k] * L + s_vals[k] * M

    return V


def spec(omega: np.ndarray, M: int, pi: Union[np.ndarray, list], spectre=100) -> np.ndarray:
    """
    Faster Python/Numpy port of the R function spec().

    The output is the same object as before: a matrix of shape (M, N).
    Speed-up comes from cumulative sums instead of thousands of repeated
    small numpy sum calls inside Python loops.
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size
    omega = np.asarray(omega, dtype=float)
    if omega.shape != (M, N):
        raise ValueError("omega must have shape (M, N)")

    mat_spectre = np.zeros((M, N), dtype=float)
    pi_down = np.sort(pi)[::-1]
    pi_prefix = np.concatenate(([0.0], np.cumsum(pi_down)))
    mu = float(pi.sum())

    if (np.isscalar(spectre) and spectre == 100) or (
        not np.isscalar(spectre)
        and len(np.atleast_1d(spectre)) == 1
        and np.atleast_1d(spectre)[0] == 100
    ):
        spectre_vec = np.zeros(M, dtype=float)
        cumsum_1 = 0.0
        lam = 0.0

        for j in range(M - 1):
            jR = j + 1
            A = max(lam, mu - cumsum_1 - (M - jR))
            B = 1.0

            for iR in range(1, M - jR + 1):
                t_idx = M - jR - iR + 1
                s_down = pi_prefix[t_idx]
                val = (mu - cumsum_1 - s_down) / iR
                if val < B:
                    B = val

            val = (mu - cumsum_1) / (M - jR + 1)
            if val < B:
                B = val

            spectre_vec[j] = A + omega[j, N - 1] * (B - A)
            lam = spectre_vec[j]
            cumsum_1 += spectre_vec[j]

        spectre_vec[M - 1] = mu - spectre_vec[: M - 1].sum()
    else:
        spectre_arr = np.asarray(spectre, dtype=float).ravel()
        if spectre_arr.size != M:
            raise ValueError("spectre must have length M")
        spectre_vec = spectre_arr

    mat_spectre[:, N - 1] = spectre_vec

    for kR in range(N - 1, 0, -1):
        startR = max(1, M - kR + 1)
        lambda1 = mat_spectre[:, kR].copy()
        lambda2 = mat_spectre[:, kR - 1].copy()
        prefix_l1 = np.concatenate(([0.0], np.cumsum(lambda1)))
        cum_l2_before_j = 0.0

        # rows before startR are zero by construction, so cum_l2_before_j starts at 0.
        for jR in range(startR, M + 1):
            lam_prev = lambda1[jR - 2] if jR >= 2 else 0.0
            sum_l1 = prefix_l1[jR]
            A = max(0.0, lam_prev, sum_l1 - cum_l2_before_j - pi_down[kR])

            B_inner = float("inf")
            for iR in range(jR, M + 1):
                start_idx0 = M - iR
                if start_idx0 < kR:
                    prem = pi_prefix[kR] - pi_prefix[start_idx0]
                else:
                    prem = 0.0

                if jR <= (iR - 1):
                    deux = prefix_l1[iR - 1] - prefix_l1[jR - 1]
                else:
                    deux = 0.0

                B_val = prem - deux - cum_l2_before_j
                if B_val < B_inner:
                    B_inner = B_val

            B = min(lambda1[jR - 1], B_inner)
            new_val = A + omega[jR - 1, kR - 1] * (B - A)
            mat_spectre[jR - 1, kR - 1] = new_val
            lambda2[jR - 1] = new_val
            cum_l2_before_j += new_val

    return mat_spectre


def CaDsd(
    pi: Union[np.ndarray, list],
    M: Optional[int] = None,
    omega: Optional[np.ndarray] = None,
    rho: Optional[np.ndarray] = None,
    spectre=100,
    U: Optional[np.ndarray] = None,
    option: bool = True,
):
    """
    Faster CaDsd implementation with the same interface and output keys.

    Main speed-ups:
    - uses the faster spec() above;
    - preallocates phi instead of repeated hstack;
    - replaces permutation matrices by direct column indexing;
    - replaces multiplication by a diagonal phase matrix with column scaling.
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size

    if M is None:
        M = int(round(pi.sum(), 7))
    if int(M) != M:
        raise ValueError("M should be an integer")
    M = int(M)

    if omega is None:
        omega = 0.5 * np.ones((M, N), dtype=float)
    else:
        omega = np.asarray(omega, dtype=float)
        if omega.shape != (M, N):
            raise ValueError("omega must have shape (M, N)")

    if rho is None:
        rho = 0.5 * np.ones((M, N - 1), dtype=float)
    else:
        rho = np.asarray(rho, dtype=float)
        if rho.shape != (M, N - 1):
            raise ValueError("rho must have shape (M, N-1)")

    rho_angles = np.round(rho * 2 * np.pi, 7)
    mat_spectre = np.round(spec(omega, M, pi, spectre), 7)
    pi_down = np.sort(pi)[::-1]

    if U is None:
        U_work = np.eye(M, dtype=complex)
    else:
        U_work = np.asarray(U, dtype=complex).copy()
        if U_work.shape != (M, M):
            raise ValueError("U must have shape (M, M)")

    phi = np.zeros((M, N), dtype=complex)
    phi[:, 0] = np.round(np.sqrt(pi_down[0]) * U_work[:, 0], 7)
    ens = np.arange(1, M + 1, dtype=int)

    eye_index = np.arange(M)

    for kR in range(2, N + 1):
        phases = np.exp(1j * rho_angles[:, kR - 2])
        lambda1 = mat_spectre[:, kR - 1].copy()
        lambda2 = mat_spectre[:, kR - 2].copy()

        E1 = ens.tolist()
        E2 = ens.tolist()

        # Keep exact equality because mat_spectre is rounded to 7 digits as in the earlier code.
        for jR in ens:
            if not E1:
                break
            val = lambda2[jR - 1]
            lam1_E1 = lambda1[np.array(E1) - 1]
            matches = np.where(lam1_E1 == val)[0]
            if matches.size > 0:
                E2 = [x for x in E2 if x != jR]
                del E1[matches[0]]

        E1_arr = np.array(E1, dtype=int)
        E2_arr = np.array(E2, dtype=int)
        E1_rev = M + 1 - E1_arr
        E2_rev = M + 1 - E2_arr
        r = len(E1_rev)

        if r == 0:
            continue

        if r != M:
            mask1 = np.ones(M, dtype=bool)
            mask2 = np.ones(M, dtype=bool)
            mask1[E1_rev - 1] = False
            mask2[E2_rev - 1] = False
            E1_perm = np.concatenate([np.sort(E1_rev), ens[mask1]])
            E2_perm = np.concatenate([np.sort(E2_rev), ens[mask2]])
            perm1 = E1_perm - 1
            perm2 = E2_perm - 1
        else:
            perm1 = eye_index
            perm2 = eye_index

        lambda2_E2 = lambda2[E2_arr - 1]
        lambda1_E1 = lambda1[E1_arr - 1]
        R = np.column_stack([lambda2_E2, lambda1_E1]).astype(complex)[::-1, :]

        v = np.zeros(r, dtype=complex)
        w = np.zeros(r, dtype=complex)

        for i in range(r):
            v1 = R[i, 0] - R[:, 1]
            v2 = R[i, 0] - R[:, 0]
            v2[i] = 1.0 + 0j
            order = np.argsort(np.abs(v1))
            v[i] = np.round(np.sqrt(-np.prod(v1[order] / v2[order])), 7)

            w1 = R[i, 1] - R[:, 0]
            w2 = R[i, 1] - R[:, 1]
            w2[i] = 1.0 + 0j
            order_w = np.argsort(np.abs(w1))
            w[i] = np.round(np.sqrt(np.prod(w1[order_w] / w2[order_w])), 7)

        col_vals = R[:, 1]
        row_vals = R[:, 0]
        denom = col_vals[np.newaxis, :] - row_vals[:, np.newaxis]
        W = (v[:, None] * w[None, :]) / denom

        # U @ diag(phases) is just column scaling.
        UV = U_work * phases[np.newaxis, :]

        # Equivalent to U %*% V %*% t(sigma2) %*% vect.
        UV_sigma2 = UV[:, perm2]
        phi[:, kR - 1] = UV_sigma2[:, :r] @ v

        # Equivalent to U %*% V %*% t(sigma2) %*% block %*% sigma1.
        U_block = UV_sigma2.copy()
        U_block[:, :r] = UV_sigma2[:, :r] @ W
        U_new = np.empty_like(U_block)
        U_new[:, perm1] = U_block
        U_work = U_new

    d = np.sqrt(mat_spectre[:, N - 1])
    if np.any(d == 0):
        raise ValueError("Zero on diagonal of spectrum, cannot invert sqrt.")

    # Avoid forming an explicit inverse diagonal matrix.
    EigenBasis = (phi.conj().T @ U_work) / d[np.newaxis, :]
    K = np.round(phi.conj().T @ phi, 7)

    return {
        "K": K,
        "spectrum": mat_spectre,
        "EigenBasis": EigenBasis,
    }

## 3. Helper Functions

In [199]:
def corr(a, b):
    return float(np.corrcoef(np.asarray(a, dtype=float), np.asarray(b, dtype=float))[0, 1])


def ht_variance_from_kernel(K, values, pi):
    values = np.asarray(values, dtype=float)
    pi = np.asarray(pi, dtype=float)
    ratio = values / pi
    A = K * (np.eye(len(pi)) - np.conjugate(K))
    return float(np.real(ratio @ (A @ ratio)))


def kernel_diagnostics(K, pi):
    eigvals = np.linalg.eigvalsh(K)
    return {
        "eig_min": float(eigvals.min()),
        "eig_max": float(eigvals.max()),
        "eig_sum": float(eigvals.sum()),
        "hermitian_error": float(np.linalg.norm(K - K.conjugate().T)),
        "idempotency_error": float(np.linalg.norm(K @ K - K)),
        "max_diag_error": float(np.max(np.abs(np.real(np.diag(K)) - pi))),
        "has_complex_entries": bool(np.max(np.abs(np.imag(K))) > 1e-10),
        "max_imag_abs": float(np.max(np.abs(np.imag(K)))),
    }


def is_valid_kernel(K, pi, strict=STRICT_VALIDATION):
    if K is None or not np.all(np.isfinite(K)):
        return False
    if not np.allclose(np.real(np.diag(K)), pi, atol=1e-3):
        return False
    if not strict:
        return True
    eigvals = np.linalg.eigvalsh(K)
    if eigvals.min() < -1e-3:
        return False
    if eigvals.max() > 1.0 + 1e-3:
        return False
    if not np.isclose(eigvals.sum(), round(pi.sum()), atol=1e-3):
        return False
    return True

## 4. Generate the 9 Scenarios You Asked For

We generate one main variable `y`, then 9 scenarios from:

- three target correlations for `z`: `z00`, `z80`, `z90`,
- three inclusion-probability cases: `pi_equal`, `pi80`, `pi90`.

For each scenario, we also impose Vincent's condition before running ABC:

- `pi` is decreasing, or equal in the equal-probability case,
- `z/pi` is monotone in the same order.

The table below checks the achieved correlations and the Vincent condition.

In [200]:
rng_population = np.random.default_rng(BASE_SEED)

Z_TARGETS = {
    "z00": 0.00,
    "z80": 0.80,
    "z90": 0.90,
}

PI_TARGETS = {
    "pi_equal": 0.00,
    "pi10": 0.10,
    "pi20": 0.20,
    "pi30": 0.30,
    "pi40": 0.40,
    "pi50": 0.50,
    "pi60": 0.60,
    "pi70": 0.70,
    "pi80": 0.80,
    "pi90": 0.90,
}

# One main variable y for all scenarios.
y_base = 100.0 + 30.0 * np.linspace(0.0, 1.0, N) + rng_population.normal(0.0, 8.0, N)


def standardize(x):
    x = np.asarray(x, dtype=float)
    sd = x.std(ddof=1)
    if sd == 0:
        return np.zeros_like(x)
    return (x - x.mean()) / sd


def orthogonal_noise(base, rng):
    base = standardize(base)
    noise = rng.normal(size=len(base))
    noise = noise - noise.mean()
    noise = noise - np.dot(noise, base) / np.dot(base, base) * base
    return standardize(noise)


def correlated_variable(y, target_corr, rng, location=100.0, scale=20.0):
    y_std = standardize(y)
    noise = orthogonal_noise(y_std, rng)
    x = target_corr * y_std + np.sqrt(max(0.0, 1.0 - target_corr**2)) * noise
    return location + scale * standardize(x)


def make_pi_from_y(y, target_corr, rng):
    if target_corr == 0.0:
        return np.full(len(y), n / len(y)), 0.0

    y_std = standardize(y)
    noise = orthogonal_noise(y_std, rng)
    score = target_corr * y_std + np.sqrt(max(0.0, 1.0 - target_corr**2)) * noise

    best = None
    for strength in np.linspace(0.05, 2.0, 160):
        size = np.exp(strength * standardize(score))
        pi = n * size / size.sum()
        if np.any(pi <= 0) or np.any(pi >= 1):
            continue
        achieved = corr(y, pi)
        err = abs(achieved - target_corr)
        if best is None or err < best[0]:
            best = (err, pi, achieved)

    if best is None:
        raise RuntimeError("Could not build pi values in (0,1).")
    return best[1], best[2]


def build_z_for_vincent(y, pi, target_corr, rng):
    """Build z so corr(y,z) is close to target and z/pi is monotone in Vincent order."""
    if np.allclose(pi, pi[0]):
        # Equal pi: make z with requested correlation, then order by z.
        z = correlated_variable(y, target_corr, rng, location=100.0, scale=20.0)
        order = np.argsort(z)  # z/pi increasing because pi is constant
        return y[order], z[order], pi[order]

    # Unequal pi: first order by decreasing pi.
    order = np.argsort(pi)[::-1]
    y_s = y[order]
    pi_s = pi[order]
    grid = np.linspace(-1.0, 1.0, len(y))

    best = None
    for direction in [1.0, -1.0]:
        for shape in ["linear", "exp", "power"]:
            for scale_value in np.linspace(0.05, 14.0, 1400):
                if shape == "linear":
                    ratio = 10.0 + direction * scale_value * grid
                elif shape == "exp":
                    ratio = 10.0 * np.exp(direction * (scale_value / 3.0) * grid)
                else:
                    u = np.linspace(0.0, 1.0, len(y))
                    base = 0.2 + u ** max(0.2, scale_value / 3.0)
                    ratio = 10.0 + direction * 6.0 * standardize(base)
                if np.min(ratio) <= 0:
                    continue
                z_s = pi_s * ratio
                z_full = np.empty_like(z_s)
                y_full = np.empty_like(y_s)
                pi_full = np.empty_like(pi_s)
                # Keep the Vincent order as the scenario order.
                y_full[:] = y_s
                z_full[:] = z_s
                pi_full[:] = pi_s
                achieved = corr(y_full, z_full)
                if not np.isfinite(achieved):
                    continue
                err = abs(achieved - target_corr)
                if best is None or err < best[0]:
                    best = (err, y_full.copy(), z_full.copy(), pi_full.copy(), achieved)

    if best is None:
        raise RuntimeError("Could not build a Vincent-compatible z.")
    return best[1], best[2], best[3]


def make_scenario(z_case, pi_case):
    rng = np.random.default_rng(BASE_SEED + 100 * list(Z_TARGETS).index(z_case) + list(PI_TARGETS).index(pi_case))
    pi_raw, achieved_pi_corr = make_pi_from_y(y_base, PI_TARGETS[pi_case], rng)
    y, z, pi = build_z_for_vincent(y_base, pi_raw, Z_TARGETS[z_case], rng)
    return pd.DataFrame({
        "unit": np.arange(N),
        "y": y,
        "z": z,
        "pi": pi,
        "y_over_pi": y / pi,
        "z_over_pi": z / pi,
    })


scenario_summary = []
for z_case in Z_TARGETS:
    for pi_case in PI_TARGETS:
        df = make_scenario(z_case, pi_case)
        pi_diff = np.diff(df["pi"])
        ratio_diff = np.diff(df["z_over_pi"])
        scenario_summary.append({
            "z_case": z_case,
            "pi_case": pi_case,
            "target_corr_y_z": Z_TARGETS[z_case],
            "target_corr_y_pi": PI_TARGETS[pi_case],
            "corr_y_z": corr(df["y"], df["z"]),
            "corr_y_pi": 0.0 if pi_case == "pi_equal" else corr(df["y"], df["pi"]),
            "corr_y_over_pi_z_over_pi": corr(df["y_over_pi"], df["z_over_pi"]),
            "pi_decreasing_or_equal": bool(np.all(pi_diff <= 1e-12)),
            "z_over_pi_monotone": bool(np.all(ratio_diff > 0) or np.all(ratio_diff < 0)),
            "pi_min": df["pi"].min(),
            "pi_max": df["pi"].max(),
        })

scenario_summary = pd.DataFrame(scenario_summary)
scenario_summary.round(4)

,z_case,pi_case,target_corr_y_z,target_corr_y_pi,corr_y_z,corr_y_pi,corr_y_over_pi_z_over_pi,pi_decreasing_or_equal,z_over_pi_monotone,pi_min,pi_max
0,z00,pi_equal,0.0,0.0,0.0000,0.0000,0.0000,True,True,0.2000,0.2000
1,z00,pi10,0.0,0.1,-0.0000,0.1080,0.4743,True,True,0.1739,0.2180
2,z00,pi20,0.0,0.2,-0.0002,0.2095,0.2336,True,True,0.1805,0.2160
3,z00,pi30,0.0,0.3,0.0060,0.3069,0.2566,True,True,0.1860,0.2213
4,z00,pi40,0.0,0.4,-0.0002,0.4001,0.9107,True,True,0.0891,0.3428
5,z00,pi50,0.0,0.5,-0.0090,0.5074,0.0588,True,True,0.1811,0.2181
6,z00,pi60,0.0,0.6,-0.0001,0.6106,0.0511,True,True,0.1814,0.2246
7,z00,pi70,0.0,0.7,-0.0016,0.7000,0.8995,True,True,0.0407,0.4287
8,z00,pi80,0.0,0.8,-0.0119,0.7980,-0.4304,True,True,0.1834,0.2239
9,z00,pi90,0.0,0.9,-0.0008,0.8979,-0.6569,True,True,0.1836,0.2278


## 5. Plot the 9 Scenarios

This is the 4-by-3 plot: top row is raw `y` vs `z`; next rows are `y/pi` vs `z/pi` under the three `pi` profiles.

In [201]:
# fig, axes = plt.subplots(4, 3, figsize=(13, 12), constrained_layout=True)

# z_cases = list(Z_TARGETS.keys())
# pi_cases = list(PI_TARGETS.keys())

# for col, z_case in enumerate(z_cases):
#     df_raw = make_scenario(z_case, "pi_equal")
#     axes[0, col].scatter(df_raw["z"], df_raw["y"], s=35, alpha=0.75)
#     axes[0, col].set_title(f"{z_case}\nr(y,z)={corr(df_raw['y'], df_raw['z']):.2f}")
#     axes[0, col].set_xlabel("z")
#     axes[0, col].set_ylabel("y")

#     for row, pi_case in enumerate(pi_cases, start=1):
#         df_plot = make_scenario(z_case, pi_case)
#         axes[row, col].scatter(df_plot["z_over_pi"], df_plot["y_over_pi"], s=35, alpha=0.75, color="tab:orange")
#         r = corr(df_plot["y_over_pi"], df_plot["z_over_pi"])
#         axes[row, col].text(
#             0.04, 0.92, f"r={r:.2f}",
#             transform=axes[row, col].transAxes,
#             ha="left", va="top", fontsize=10,
#             bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none", "pad": 2},
#         )
#         axes[row, col].set_xlabel("z / pi")
#         axes[row, col].set_ylabel("y / pi")

# row_labels = ["raw", *pi_cases]
# for row, label in enumerate(row_labels):
#     axes[row, 0].annotate(
#         label,
#         xy=(-0.25, 0.5),
#         xycoords="axes fraction",
#         rotation=90,
#         va="center",
#         ha="center",
#         fontsize=11,
#         fontweight="bold",
#     )

# fig.suptitle("Nine scenarios: z targets crossed with pi targets", fontsize=14)
# plt.show()

## 6. Scenario Evaluation Function

For every scenario, efficiency is relative to the Vincent/P$\pi$ reference:

$$	ext{eff}_z = V_z^{Vincent} / V_z^{candidate}.$$

So values above 1 mean the candidate has smaller HT variance than Vincent/P$\pi$.

In [202]:
@dataclass
class Food:
    omega: np.ndarray
    rho: np.ndarray
    result: dict
    trial: int = 0

    @property
    def score(self):
        return self.result["eff_z"]


def run_one_scenario(z_case, pi_case, verbose=True):
    df = make_scenario(z_case, pi_case)
    y = df["y"].to_numpy()
    z = df["z"].to_numpy()
    pi = df["pi"].to_numpy()

    if not np.all(np.diff(pi) <= 1e-12):
        raise ValueError(f"{z_case}/{pi_case}: pi is not decreasing/equal")
    ratio_diff = np.diff(z / pi)
    if not (np.all(ratio_diff > 0) or np.all(ratio_diff < 0)):
        raise ValueError(f"{z_case}/{pi_case}: z/pi is not monotone")

    M = n

    # Vincent/Ppi reference for z: use the current z/pi monotone order.
    K_vincent_z = Ppi(pi) @ Ppi(pi).T
    omega_vincent = np.zeros((M, N))
    rho_vincent = 0.5 * np.ones((M, N - 1))
    var_z_vincent = ht_variance_from_kernel(K_vincent_z, z, pi)

    # Vincent/Ppi reference for y: build a separate y/pi monotone order.
    # This is the denominator for y efficiency. Therefore the z-optimal start
    # is not automatically 1 for y.
    y_order = np.argsort(y / pi)
    y_for_y_ref = y[y_order]
    pi_for_y_ref = pi[y_order]
    K_vincent_y = Ppi(pi_for_y_ref) @ Ppi(pi_for_y_ref).T
    var_y_vincent = ht_variance_from_kernel(K_vincent_y, y_for_y_ref, pi_for_y_ref)

    def evaluate(omega, rho):
        try:
            K = CaDsd(pi=pi, M=M, omega=omega, rho=rho)["K"]
        except Exception:
            return {"valid": False, "eff_z": 0.0, "eff_y": 0.0, "var_z": np.inf, "var_y": np.inf, "K": None}
        if not is_valid_kernel(K, pi, strict=STRICT_VALIDATION):
            return {"valid": False, "eff_z": 0.0, "eff_y": 0.0, "var_z": np.inf, "var_y": np.inf, "K": K}
        var_z = ht_variance_from_kernel(K, z, pi)
        var_y = ht_variance_from_kernel(K, y, pi)
        if var_z <= 0 or var_y <= 0 or not np.isfinite(var_z) or not np.isfinite(var_y):
            return {"valid": False, "eff_z": 0.0, "eff_y": 0.0, "var_z": var_z, "var_y": var_y, "K": K}
        return {
            "valid": True,
            "eff_z": var_z_vincent / var_z,
            "eff_y": var_y_vincent / var_y,
            "var_z": var_z,
            "var_y": var_y,
            "K": K,
        }

    start_result = evaluate(omega_vincent, rho_vincent)
    start_matches_ppi = bool(start_result["valid"] and abs(start_result["eff_z"] - 1.0) <= 0.002)
    if not start_matches_ppi and verbose:
        print(
            f"WARNING {z_case}/{pi_case}: CaDSD warm start does not match Ppi; "
            f"Start_z/V={start_result['eff_z']:.4f}. Continuing and reporting it."
        )

    rng = np.random.default_rng(BASE_SEED + 100 * list(Z_TARGETS).index(z_case) + list(PI_TARGETS).index(pi_case))
    random_rng = np.random.default_rng(BASE_SEED + 999 + 100 * list(Z_TARGETS).index(z_case) + list(PI_TARGETS).index(pi_case))

    def random_candidate():
        return rng.random((M, N)), rng.random((M, N - 1))

    def random_candidate_for_random_search():
        return random_rng.random((M, N)), random_rng.random((M, N - 1))

    def near_candidate(omega_center, rho_center, scale):
        omega = omega_center + rng.normal(0.0, scale, omega_center.shape)
        rho = rho_center + rng.normal(0.0, scale, rho_center.shape)
        return np.clip(omega, 0.0, 1.0), np.clip(rho, 0.0, 1.0)

    def make_food(omega, rho):
        result = evaluate(omega, rho)
        if not result["valid"]:
            return None
        return Food(omega=omega, rho=rho, result=result)

    # Choose start.
    if START_MODE == "vincent":
        start_omega = omega_vincent.copy()
        start_rho = rho_vincent.copy()
        selected_start = start_result
        start_evaluations = 1
    elif START_MODE == "random_best":
        selected_start = None
        start_omega = None
        start_rho = None
        start_evaluations = RANDOM_START_CANDIDATES
        for _ in range(RANDOM_START_CANDIDATES):
            omega, rho = random_candidate()
            result = evaluate(omega, rho)
            if result["valid"] and (selected_start is None or result["eff_z"] > selected_start["eff_z"]):
                selected_start = result
                start_omega = omega
                start_rho = rho
        if selected_start is None:
            raise RuntimeError(f"{z_case}/{pi_case}: no valid random start found")
    else:
        raise ValueError("START_MODE must be 'vincent' or 'random_best'")

    # Initialize ABC population.
    population = []
    start_food = make_food(start_omega, start_rho)
    population.append(start_food)
    attempts = 0
    while len(population) < COLONY_SIZE and attempts < COLONY_SIZE * 100:
        if rng.random() < RANDOM_GLOBAL_PROBABILITY:
            omega, rho = random_candidate()
        else:
            omega, rho = near_candidate(start_omega, start_rho, INITIAL_NEAR_START_SCALE)
        food = make_food(omega, rho)
        if food is not None:
            population.append(food)
        attempts += 1

    best_food = max(population, key=lambda f: f.score)
    random_best = make_food(start_omega, start_rho)
    abc_evaluations = start_evaluations + len(population)
    random_evaluations = 1
    records = []

    def mutate(food, partner, best, progress, mode="abc"):
        if mode == "local":
            scale = max(0.004, 0.04 * (1.0 - progress))
            return near_candidate(best.omega, best.rho, scale)
        adaptive_scale = max(0.08, 1.0 - 0.90 * progress)
        omega = food.omega + rng.uniform(-adaptive_scale, adaptive_scale, food.omega.shape) * (food.omega - partner.omega)
        rho = food.rho + rng.uniform(-adaptive_scale, adaptive_scale, food.rho.shape) * (food.rho - partner.rho)
        if progress > 0.10:
            pull = rng.uniform(0.0, 0.25 * progress)
            omega = omega + pull * (best.omega - omega)
            rho = rho + pull * (best.rho - rho)
        jitter = 0.004 * adaptive_scale
        omega = omega + rng.normal(0.0, jitter, omega.shape)
        rho = rho + rng.normal(0.0, jitter, rho.shape)
        return np.clip(omega, 0.0, 1.0), np.clip(rho, 0.0, 1.0)

    if verbose:
        print(f"Running {z_case} / {pi_case}")

    threshold_reached = False
    stop_reason = "max_iterations"

    for iteration in range(1, MAX_ITERATIONS + 1):
        progress = iteration / MAX_ITERATIONS
        evals_this_iteration = 0

        # Employed bees.
        new_population = []
        for i, food in enumerate(population):
            j = rng.integers(0, len(population) - 1)
            if j >= i:
                j += 1
            omega, rho = mutate(food, population[j], best_food, progress)
            candidate = make_food(omega, rho)
            evals_this_iteration += 1
            if candidate is not None and candidate.score > food.score:
                new_population.append(candidate)
                if candidate.score > best_food.score:
                    best_food = candidate
            else:
                food.trial += 1
                new_population.append(food)
        population = new_population

        # Onlooker bees.
        scores = np.array([f.score for f in population])
        weights = scores - scores.min() + 1e-12
        probs = weights / weights.sum()
        n_onlookers = max(1, int(round(ONLOOKER_FACTOR * COLONY_SIZE)))
        for _ in range(n_onlookers):
            i = int(rng.choice(len(population), p=probs))
            j = rng.integers(0, len(population) - 1)
            if j >= i:
                j += 1
            mode = "local" if rng.random() < 0.10 else "abc"
            omega, rho = mutate(population[i], population[j], best_food, progress, mode=mode)
            candidate = make_food(omega, rho)
            evals_this_iteration += 1
            if candidate is not None and candidate.score > population[i].score:
                population[i] = candidate
                if candidate.score > best_food.score:
                    best_food = candidate
            else:
                population[i].trial += 1

        # Scouts.
        for i, food in enumerate(population):
            if food.trial >= LIMIT:
                if rng.random() < 0.60:
                    omega, rho = near_candidate(best_food.omega, best_food.rho, 0.05)
                else:
                    omega, rho = random_candidate()
                candidate = make_food(omega, rho)
                evals_this_iteration += 1
                if candidate is not None:
                    population[i] = candidate
                    if candidate.score > best_food.score:
                        best_food = candidate
                else:
                    food.trial = 0

        # Local search.
        if LOCAL_SEARCH_INTERVAL and iteration % LOCAL_SEARCH_INTERVAL == 0:
            for _ in range(LOCAL_SEARCH_ATTEMPTS):
                omega, rho = mutate(best_food, best_food, best_food, progress, mode="local")
                candidate = make_food(omega, rho)
                evals_this_iteration += 1
                if candidate is not None and candidate.score > best_food.score:
                    best_food = candidate

        abc_evaluations += evals_this_iteration

        # Random search gets the same effort.
        for _ in range(evals_this_iteration):
            omega, rho = random_candidate_for_random_search()
            candidate = make_food(omega, rho)
            random_evaluations += 1
            if candidate is not None and candidate.score > random_best.score:
                random_best = candidate

        threshold_reached = bool(best_food.score >= NEEDED_THRESHOLD)
        if threshold_reached:
            stop_reason = "threshold_reached"

        should_report = (
            iteration == 1
            or iteration % CHECKPOINT_INTERVAL == 0
            or iteration == MAX_ITERATIONS
            or threshold_reached
        )

        if should_report:
            row = {
                "z_case": z_case,
                "pi_case": pi_case,
                "iteration": iteration,
                "corr_y_z": corr(y, z),
                "corr_y_pi": 0.0 if pi_case == "pi_equal" else corr(y, pi),
                "corr_y_over_pi_z_over_pi": corr(y / pi, z / pi),
                "Vincent_start_matches_Ppi": start_matches_ppi,
                "Start_z_over_V": start_result["eff_z"],
                "Start_y_over_Vy": start_result["eff_y"],
                "Selected_start_z_over_V": selected_start["eff_z"],
                "Selected_start_y_over_Vy": selected_start["eff_y"],
                "ABC_z_over_V": best_food.result["eff_z"],
                "ABC_y_over_Vy": best_food.result["eff_y"],
                "Random_z_over_V": random_best.result["eff_z"],
                "Random_y_over_Vy": random_best.result["eff_y"],
                "ABC_evaluations": abc_evaluations,
                "Random_evaluations": random_evaluations,
                "Needed_threshold": NEEDED_THRESHOLD,
                "Stop_reason": stop_reason,
            }
            records.append(row)
            if verbose:
                print(
                    f"iter={iteration:4d} "
                    f"ABC_z/V={row['ABC_z_over_V']:.3f} "
                    f"ABC_y/Vy={row['ABC_y_over_Vy']:.3f} "
                    f"Rnd_z/V={row['Random_z_over_V']:.3f} "
                    f"Rnd_y/Vy={row['Random_y_over_Vy']:.3f} "
                    f"stop={row['Stop_reason']}"
                )

        if threshold_reached:
            break

    return records


## 7. Run the 9 Scenarios

This prints results every `CHECKPOINT_INTERVAL` iterations, and also immediately when the optimized-variable efficiency reaches `NEEDED_THRESHOLD` (`ABC_z/V` in the notebook output; this is the study-variable efficiency in the paper notation). Each scenario stops at the threshold or at `MAX_ITERATIONS`.

In [203]:
all_rows = []
for z_case in Z_TARGETS:
    for pi_case in PI_TARGETS:
        all_rows.extend(run_one_scenario(z_case, pi_case, verbose=True))

checkpoint_table = pd.DataFrame(all_rows)
final_table = (
    checkpoint_table
    .sort_values(["z_case", "pi_case", "iteration"])
    .groupby(["z_case", "pi_case"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

final_table.round(4)

Running z00 / pi_equal
iter=   1 ABC_z/V=1.000 ABC_y/Vy=0.162 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter=  50 ABC_z/V=1.019 ABC_y/Vy=0.168 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 100 ABC_z/V=1.019 ABC_y/Vy=0.168 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 150 ABC_z/V=1.019 ABC_y/Vy=0.168 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 200 ABC_z/V=1.028 ABC_y/Vy=0.170 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 250 ABC_z/V=1.029 ABC_y/Vy=0.167 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 300 ABC_z/V=1.051 ABC_y/Vy=0.171 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 350 ABC_z/V=1.051 ABC_y/Vy=0.171 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 400 ABC_z/V=1.051 ABC_y/Vy=0.171 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 450 ABC_z/V=1.051 ABC_y/Vy=0.171 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 500 ABC_z/V=1.055 ABC_y/Vy=0.170 Rnd_z/V=1.000 Rnd_y/Vy=0.162 stop=max_iterations
iter= 550

,z_case,pi_case,iteration,corr_y_z,corr_y_pi,corr_y_over_pi_z_over_pi,Vincent_start_matches_Ppi,Start_z_over_V,Start_y_over_Vy,Selected_start_z_over_V,Selected_start_y_over_Vy,ABC_z_over_V,ABC_y_over_Vy,Random_z_over_V,Random_y_over_Vy,ABC_evaluations,Random_evaluations,Needed_threshold,Stop_reason
0,z00,pi10,1000,-0.0000,0.1080,0.4743,True,1.0000,0.1466,1.0000,0.1466,1.0118,0.1533,1.0000,0.1466,15282,15272,3.5,max_iterations
1,z00,pi20,4,-0.0002,0.2095,0.2336,False,0.9957,0.1407,0.9957,0.1407,12.7073,0.1386,0.9957,0.1407,71,61,3.5,threshold_reached
2,z00,pi30,2,0.0060,0.3069,0.2566,True,1.0002,0.2507,1.0002,0.2507,9.9724,0.2571,1.0002,0.2507,41,31,3.5,threshold_reached
3,z00,pi40,1000,-0.0002,0.4001,0.9107,True,1.0000,0.9328,1.0000,0.9328,1.0018,0.9315,1.0000,0.9328,15274,15264,3.5,max_iterations
4,z00,pi50,49,-0.0090,0.5074,0.0588,True,1.0005,0.2174,1.0005,0.2174,29.7151,0.2182,1.0005,0.2174,759,749,3.5,threshold_reached
5,z00,pi60,12,-0.0001,0.6106,0.0511,True,1.0005,0.3615,1.0005,0.3615,3.9158,0.3751,1.0005,0.3615,191,181,3.5,threshold_reached
6,z00,pi70,1000,-0.0016,0.7000,0.8995,True,0.9999,0.9859,0.9999,0.9859,1.0031,0.9866,0.9999,0.9859,15234,15224,3.5,max_iterations
7,z00,pi80,11,-0.0119,0.7980,-0.4304,True,0.9996,0.4431,0.9996,0.4431,5.8407,0.4740,0.9996,0.4431,176,166,3.5,threshold_reached
8,z00,pi90,4,-0.0008,0.8979,-0.6569,True,0.9995,0.2320,0.9995,0.2320,5.1005,0.2525,0.9995,0.2320,71,61,3.5,threshold_reached
9,z00,pi_equal,1000,0.0000,0.0000,0.0000,True,1.0000,0.1617,1.0000,0.1617,1.0672,0.1690,1.0000,0.1617,15322,15312,3.5,max_iterations


## 8. Final Compact Table

In [204]:
compact_columns = [
    "z_case", "pi_case", "corr_y_z", "corr_y_pi", "corr_y_over_pi_z_over_pi",
    "Vincent_start_matches_Ppi",
    "Start_z_over_V", "Start_y_over_Vy",
    "Selected_start_z_over_V", "Selected_start_y_over_Vy",
    "ABC_z_over_V", "ABC_y_over_Vy",
    "Random_z_over_V", "Random_y_over_Vy",
    "ABC_evaluations", "Random_evaluations",
]

final_table[compact_columns].round(3)


,z_case,pi_case,corr_y_z,corr_y_pi,corr_y_over_pi_z_over_pi,Vincent_start_matches_Ppi,Start_z_over_V,Start_y_over_Vy,Selected_start_z_over_V,Selected_start_y_over_Vy,ABC_z_over_V,ABC_y_over_Vy,Random_z_over_V,Random_y_over_Vy,ABC_evaluations,Random_evaluations
0,z00,pi10,-0.000,0.108,0.474,True,1.000,0.147,1.000,0.147,1.012,0.153,1.000,0.147,15282,15272
1,z00,pi20,-0.000,0.209,0.234,False,0.996,0.141,0.996,0.141,12.707,0.139,0.996,0.141,71,61
2,z00,pi30,0.006,0.307,0.257,True,1.000,0.251,1.000,0.251,9.972,0.257,1.000,0.251,41,31
3,z00,pi40,-0.000,0.400,0.911,True,1.000,0.933,1.000,0.933,1.002,0.932,1.000,0.933,15274,15264
4,z00,pi50,-0.009,0.507,0.059,True,1.001,0.217,1.001,0.217,29.715,0.218,1.001,0.217,759,749
5,z00,pi60,-0.000,0.611,0.051,True,1.000,0.362,1.000,0.362,3.916,0.375,1.000,0.362,191,181
6,z00,pi70,-0.002,0.700,0.899,True,1.000,0.986,1.000,0.986,1.003,0.987,1.000,0.986,15234,15224
7,z00,pi80,-0.012,0.798,-0.430,True,1.000,0.443,1.000,0.443,5.841,0.474,1.000,0.443,176,166
8,z00,pi90,-0.001,0.898,-0.657,True,1.000,0.232,1.000,0.232,5.100,0.252,1.000,0.232,71,61
9,z00,pi_equal,0.000,0.000,0.000,True,1.000,0.162,1.000,0.162,1.067,0.169,1.000,0.162,15322,15312


## 9. Checkpoint Table

Use this if you want to plot or inspect the path of the search.

In [205]:
checkpoint_table.round(3)

,z_case,pi_case,iteration,corr_y_z,corr_y_pi,corr_y_over_pi_z_over_pi,Vincent_start_matches_Ppi,Start_z_over_V,Start_y_over_Vy,Selected_start_z_over_V,Selected_start_y_over_Vy,ABC_z_over_V,ABC_y_over_Vy,Random_z_over_V,Random_y_over_Vy,ABC_evaluations,Random_evaluations,Needed_threshold,Stop_reason
0,z00,pi_equal,1,0.0,0.0,0.00,True,1.0,0.162,1.0,0.162,1.000,0.162,1.0,0.162,26,16,3.5,max_iterations
1,z00,pi_equal,50,0.0,0.0,0.00,True,1.0,0.162,1.0,0.162,1.019,0.168,1.0,0.162,772,762,3.5,max_iterations
2,z00,pi_equal,100,0.0,0.0,0.00,True,1.0,0.162,1.0,0.162,1.019,0.168,1.0,0.162,1536,1526,3.5,max_iterations
3,z00,pi_equal,150,0.0,0.0,0.00,True,1.0,0.162,1.0,0.162,1.019,0.168,1.0,0.162,2300,2290,3.5,max_iterations
4,z00,pi_equal,200,0.0,0.0,0.00,True,1.0,0.162,1.0,0.162,1.028,0.170,1.0,0.162,3064,3054,3.5,max_iterations
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,z90,pi90,800,0.9,0.9,-0.83,True,1.0,0.500,1.0,0.500,1.025,0.545,1.0,0.500,12223,12213,3.5,max_iterations
377,z90,pi90,850,0.9,0.9,-0.83,True,1.0,0.500,1.0,0.500,1.025,0.545,1.0,0.500,12988,12978,3.5,max_iterations
378,z90,pi90,900,0.9,0.9,-0.83,True,1.0,0.500,1.0,0.500,1.034,0.545,1.0,0.500,13755,13745,3.5,max_iterations
379,z90,pi90,950,0.9,0.9,-0.83,True,1.0,0.500,1.0,0.500,1.034,0.545,1.0,0.500,14519,14509,3.5,max_iterations
